In [ ]:
from google.colab import files
import io
import pandas as pd
import json
import re

# 1. Upload the CSV
# Press play in the top left and then follow the prompt below to upload your .csv
print("Step 1: Please upload your publication CSV file.")
uploaded = files.upload()

if not uploaded:
    print("No file uploaded. Please run this cell again to try again.")
else:
    fn = list(uploaded.keys())[0]
    print(f"Successfully received: {fn}")

In [ ]:
# 2. Process the Data
# Press the play icon on the left once you have uploaded .csv
if 'uploaded' in locals() and uploaded:
    fn = list(uploaded.keys())[0]
    df = pd.read_csv(io.BytesIO(uploaded[fn]))

    # Normalize column names
    df.columns = df.columns.str.lower().str.replace(' ', '-')

    # Define stop words
    stop_words = {'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for', 'of', 'with', 'by', 'from', 'is','was', 'were', 'are', 'than', 'this', 'that'}

    word_index = {}

    for _, row in df.iterrows():
        # Flexible column selection
        title = str(row.get('article-title', row.get('title', 'Untitled')))
        doi = str(row.get('doi', ''))

        # Extract words (3+ letters)
        words = re.findall(r'\b[A-Za-z]{3,}\b', title.lower())

        for word in words:
            if word in stop_words:
                continue

            # Stemming logic
            root = word
            if word.endswith('s') and not word.endswith(('is', 'us', 'ss')):
                if word.endswith('ies'):
                    root = word[:-3] + 'y'
                elif word.endswith('es'):
                    root = word[:-2]
                else:
                    root = word[:-1]

            if root not in word_index:
                word_index[root] = []

            # Safety check and limit
            if len(word_index[root]) < 5:
                citation = {"title": title, "doi": doi}
                if citation not in word_index[root]:
                    word_index[root].append(citation)

    print(f"Successfully processed {len(word_index)} unique word tiles.")
else:
    print("Error: No data found. Please run Cell 1 first.")

In [ ]:
# 3. Save and Download
# Press the play icon on the left, the file will automatically download as word_data.json.
if 'word_index' in locals():
    output_filename = 'word_data.json'
    with open(output_filename, 'w') as f:
        json.dump(word_index, f, indent=2)

    print(f"File '{output_filename}' is ready.")
    files.download(output_filename)
else:
    print("Error: Nothing to save. Please run the previous cells first.")